In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.


In [2]:
## 1: Load the datasets into pandas DataFrames.

In [3]:
bradenton = pd.read_csv("../data/bradentonFL_tempData.csv")
cyanotoxins = pd.read_csv("../data/florida_cyanotoxins_2015-01-01_to_2026-06-17.csv")
habsos = pd.read_csv("../data/habsos_cellcounts.csv")
mote = pd.read_csv("../data/MoteMarine_BottomTemp.csv")
nutrients = pd.read_csv("../data/NutrientsFL_2006_2025.csv")
sarasota = pd.read_csv("../data/Sarasota_Wind&Temp.csv")

/var/folders/mq/c7lsj5l90m56k106b4gsl9_r0000gp/T/ipykernel_14856/3789633189.py:2: DtypeWarning: Columns (48) have mixed types. Specify dtype option on import or set low_memory=False.
  cyanotoxins = pd.read_csv("../data/florida_cyanotoxins_2015-01-01_to_2026-06-17.csv")
/var/folders/mq/c7lsj5l90m56k106b4gsl9_r0000gp/T/ipykernel_14856/3789633189.py:3: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  habsos = pd.read_csv("../data/habsos_cellcounts.csv")
/var/folders/mq/c7lsj5l90m56k106b4gsl9_r0000gp/T/ipykernel_14856/3789633189.py:6: DtypeWarning: Columns (1,2,4,6,8,10,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  sarasota = pd.read_csv("../data/Sarasota_Wind&Temp.csv")


In [4]:
## 2: Prepare the datasets: Identify categorical and numerical columns. 

BradentonFL_tempData identifying categorical and numerical columns. 

In [5]:
bradenton.head()

,COOPID,YEAR,MONTH,DAY,PRECIPITATION,MAX TEMP,MIN TEMP,MEAN TEMP
0,80945,2015,1,1,0.0,74.0,60.0,
1,80945,2015,1,2,0.0,78.0,64.0,
2,80945,2015,1,3,0.0,85.0,70.0,
3,80945,2015,1,4,0.0,83.0,69.0,
4,80945,2015,1,5,0.0,75.0,61.0,


In [6]:
bradenton.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4018 entries, 0 to 4017
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   COOPID          4018 non-null   int64  
 1    YEAR           4018 non-null   int64  
 2    MONTH          4018 non-null   int64  
 3    DAY            4018 non-null   int64  
 4    PRECIPITATION  4018 non-null   float64
 5    MAX TEMP       4018 non-null   float64
 6    MIN TEMP       4018 non-null   float64
 7    MEAN TEMP      4018 non-null   object 
dtypes: float64(3), int64(4), object(1)
memory usage: 251.3+ KB


Bradenton CSV has extra spaces in the column names, therefore we must remove extra whitespace from the beginning and end of each column name.

In [7]:
bradenton.columns = bradenton.columns.str.strip()

In [8]:
bradenton["MEAN TEMP"].value_counts(dropna=False).head(10)

MEAN TEMP
             731
 84.00000    131
 85.00000    126
 84.50000    108
 83.50000    104
 83.00000    102
 85.50000     90
 82.00000     85
 86.00000     83
 75.00000     78
Name: count, dtype: int64

Found 731 missing MEAN TEMP values

In [9]:
bradenton_numeric = [
    "PRECIPITATION",
    "MAX TEMP",
    "MIN TEMP",
    "MEAN TEMP"
]

Cyanotoxins Dataset: Identifying numerical and categorical columns

In [10]:
cyanotoxins.head()

,Org_Identifier,Org_FormalName,Project_Identifier,Location_Identifier,Location_Name,Location_Type,Location_State,Location_HUCEightDigitCode,Location_HUCTwelveDigitCode,Location_TribalLand,...,Result_DetectionLimitProfileDow,Result_LabSamplePrepProfileDown,ProviderName,Result_CharacteristicComparable,Result_CharacteristicGroup,Org_Type,LastChangeDate,USGSpcode,ObjectId,Value_control
0,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10246,NCA_FL-10246,Ocean,Florida,NaN,NaN,NaN,...,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,23431,A
1,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10245,NCA_FL-10245,Ocean,Florida,NaN,NaN,NaN,...,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,22625,A
2,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10297,NCA_FL-10297,Ocean,Florida,NaN,NaN,NaN,...,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,22713,A
3,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10299,NCA_FL-10299,Ocean,Florida,NaN,NaN,NaN,...,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,25340,A
4,NARS_WQX,EPA National Aquatic Resources Survey (NARS),"[""NARS_NCCA2015""]",NARS_WQX-NCA_FL-10248,NCA_FL-10248,Ocean,Florida,NaN,NaN,NaN,...,https://www.waterqualitydata.us/data/providers...,NaN,STORET,NaN,"Cyanotoxins, Phytotoxins",Federal/US Government,Fri Jan 31 08:59:46 GMT 2025,NaN,25355,A


In [11]:
cyanotoxins.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98884 entries, 0 to 98883
Data columns (total 65 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Org_Identifier                   98884 non-null  object 
 1   Org_FormalName                   98884 non-null  object 
 2   Project_Identifier               98884 non-null  object 
 3   Location_Identifier              98884 non-null  object 
 4   Location_Name                    98884 non-null  object 
 5   Location_Type                    98884 non-null  object 
 6   Location_State                   98884 non-null  object 
 7   Location_HUCEightDigitCode       98696 non-null  float64
 8   Location_HUCTwelveDigitCode      98576 non-null  float64
 9   Location_TribalLand              0 non-null      float64
 10  Location_Latitude                98884 non-null  float64
 11  Location_Longitude               98884 non-null  float64
 12  Activity_ActivityI

In [12]:
cyanotoxins["Activity_Media"].value_counts(dropna=False)

Activity_Media
Water    98884
Name: count, dtype: int64

This will be excluded since it contains only one unique value

In [13]:
cyanotoxins[[
    "Result_Characteristic",
    "Reslt_Measre",
    "Result_MeasureUnit"
]].head(20)

,Result_Characteristic,Reslt_Measre,Result_MeasureUnit
0,Microcystin,NaN,ug/L
1,Microcystin,NaN,ug/L
2,Microcystin,NaN,ug/L
3,Microcystin,NaN,ug/L
4,Microcystin,NaN,ug/L
5,Microcystin,NaN,ug/L
6,Microcystin,NaN,ug/L
7,Microcystin,NaN,ug/L
8,Microcystin,NaN,ug/L
9,Microcystin,NaN,ug/L


In [14]:
cyanotoxins_numeric = [
    "Location_Latitude",
    "Location_Longitude",
    "Activity_DepthHeightMeasure",
    "ResultDepthHeight_Measure",
    "DetectionLimit_MeasureA",
    "DetectionLimit_MeasureB",
    "Reslt_Measre"
]

In [15]:
cyanotoxins_categorical = [
    "Location_Type",
    "Activity_MediaSubdivision",
    "Activity_DepthHeightMeasureUnit",
    "SampleCollectionMethod_Name",
    "Result_Characteristic",
    "Result_SampleFraction",
    "Result_MeasureValueType",
    "Org_Type"
]

Result_Characteristic is especially important because it tells us what cyanotoxin is being measured

In [16]:
cyanotoxins.loc[
    cyanotoxins["Reslt_Measre"].notna(),
    ["Result_Characteristic", "Reslt_Measre", "Result_MeasureUnit"]
].value_counts().head(20)

Result_Characteristic      Reslt_Measre  Result_MeasureUnit
Microcystin YR             0.25          ug/L                  7009
Anatoxin-A                 0.25          ug/L                  7006
Microcystin LW             0.25          ug/L                  6353
Microcystin WR             0.50          ug/L                  6282
Microcystin LR, Desmethyl  0.25          ug/L                  6111
Microcystin HtyR           0.25          ug/L                  5346
Microcystin HilR           0.25          ug/L                  5262
Nodularin                  0.10          ug/L                  5170
Microcystin LF             0.10          ug/L                  5091
Microcystin LY             0.10          ug/L                  5088
Microcystin RR             0.10          ug/L                  4905
Cylindrospermopsin         0.10          ug/L                  4878
Microcystin LR             0.25          ug/L                  4817
Microcystin LA             0.10          ug/L           

Habos

In [17]:
habsos.head()

,OBJECTID,DESCRIPTION,LATITUDE,LONGITUDE,STATE_ID,SAMPLE_DATE,SAMPLE_DEPTH,GENUS,SPECIES,CATEGORY,...,WATER_TEMP,WATER_TEMP_UNIT,WATER_TEMP_QA,WIND_DIR,WIND_DIR_UNIT,WIND_DIR_QA,WIND_SPEED,WIND_SPEED_UNIT,WIND_SPEED_QA,QA_COMMENT
0,1,Tom Adams Bridge (Lemon Bay),26.93450,-82.3535,FL,1953-08-19 00:00:00,0.5,Karenia,brevis,medium,...,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
1,2,Naples Pier,26.13163,-81.8063,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,not observed,...,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
2,3,Piney Point; Lee County,26.53340,-81.9796,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,not observed,...,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
3,4,Cortez Bridge,27.46840,-82.6937,FL,1953-08-26 00:00:00,0.5,Karenia,brevis,medium,...,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day
4,5,Sarasota Bay; 3 mi North of bridge,27.37750,-82.5708,FL,1953-09-03 00:00:00,0.5,Karenia,brevis,high,...,NaN,NaN,9,NaN,NaN,9,NaN,NaN,9,Missing/invalid time of day


In [18]:
habsos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219152 entries, 0 to 219151
Data columns (total 26 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   OBJECTID         219152 non-null  int64  
 1   DESCRIPTION      216064 non-null  object 
 2   LATITUDE         219152 non-null  float64
 3   LONGITUDE        219152 non-null  float64
 4   STATE_ID         219152 non-null  object 
 5   SAMPLE_DATE      219152 non-null  object 
 6   SAMPLE_DEPTH     212932 non-null  float64
 7   GENUS            219152 non-null  object 
 8   SPECIES          219152 non-null  object 
 9   CATEGORY         218296 non-null  object 
 10  CELLCOUNT        219152 non-null  float64
 11  CELLCOUNT_UNIT   219152 non-null  object 
 12  CELLCOUNT_QA     219152 non-null  int64  
 13  SALINITY         113544 non-null  float64
 14  SALINITY_UNIT    113544 non-null  object 
 15  SALINITY_QA      219152 non-null  int64  
 16  WATER_TEMP       112007 non-null  floa

In [19]:
for col in ["STATE_ID", "GENUS", "SPECIES", "CATEGORY"]:
    print("\n", col)
    print(habsos[col].nunique(dropna=False), "unique values")
    print(habsos[col].value_counts(dropna=False).head(10))


 STATE_ID
4 unique values
STATE_ID
FL    211838
TX      3139
AL      3125
MS      1050
Name: count, dtype: int64

 GENUS
1 unique values
GENUS
Karenia    219152
Name: count, dtype: int64

 SPECIES
1 unique values
SPECIES
brevis    219152
Name: count, dtype: int64

 CATEGORY
6 unique values
CATEGORY
not observed    172626
very low         18778
low              12226
medium           11207
high              3459
NaN                856
Name: count, dtype: int64


I checked the categorical variables and found that STATE_ID contains four different states, making it a useful categorical feature to encode. GENUS and SPECIES each contain only one unique value, while CATEGORY contains multiple meaningful HAB risk categories and may be used as the target variable

In [20]:
habsos_categorical = [
    "STATE_ID"
]

In [21]:
habsos_numeric = [
    "LATITUDE",
    "LONGITUDE",
    "SAMPLE_DEPTH",
    "CELLCOUNT",
    "SALINITY",
    "WATER_TEMP",
    "WIND_SPEED"
]

Mote

In [22]:
mote.head()

,time,latitude,longitude,z,sea_water_temperature,sea_water_temperature_qc_agg
0,UTC,degrees_north,degrees_east,m,degree_Celsius,NaN
1,2020-08-13T15:00:00Z,27.3292,-82.5564,-2.0,30.929,1.0
2,2020-08-13T16:00:00Z,27.3292,-82.5564,-2.0,31.073,1.0
3,2020-08-13T17:00:00Z,27.3292,-82.5564,-2.0,31.217,1.0
4,2020-08-13T18:00:00Z,27.3292,-82.5564,-2.0,31.274,1.0


In [23]:
mote.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31730 entries, 0 to 31729
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   time                          31730 non-null  object 
 1   latitude                      31730 non-null  object 
 2   longitude                     31730 non-null  object 
 3   z                             31730 non-null  object 
 4   sea_water_temperature         31730 non-null  object 
 5   sea_water_temperature_qc_agg  31729 non-null  float64
dtypes: float64(1), object(5)
memory usage: 1.5+ MB


In [24]:
mote = mote.iloc[1:].copy()

Removed the first row because it contained measurement units rather than actual observations, ensuring the dataset only contains usable data.

Converted the measurement columns from text to numeric values so they could be properly analyzed and considered for numerical preprocessing.

In [25]:
mote["latitude"] = pd.to_numeric(mote["latitude"])
mote["longitude"] = pd.to_numeric(mote["longitude"])
mote["z"] = pd.to_numeric(mote["z"])
mote["sea_water_temperature"] = pd.to_numeric(mote["sea_water_temperature"])

In [26]:
mote[["latitude", "longitude", "z", "sea_water_temperature"]].nunique()

latitude                   1
longitude                  1
z                          1
sea_water_temperature    876
dtype: int64

I used nunique() to identify which numerical columns had meaningful variation. Latitude, longitude, and depth were constant, while sea water temperature varied across observations and was selected for normalization.

In [27]:
mote_numeric = [
    "sea_water_temperature"
]

Based on the variation check, we selected sea_water_temperature as the only numerical feature to be standardized because the other numerical columns were constant.

No categorical variables were selected for Mote because the dataset primarily contains measurements, timestamps, and a quality-control flag

NutrientsFL_2006_2025.csv 

In [28]:
nutrients.head()

,DataSource,StationID,Actual_DataSource,Actual_StationID,Activity_Start_Date,Activity_Start_Time,Activity_Type,RelativeDepth,Activity_Depth,Activity_Depth_Unit,Characteristic,Result_Value,Result_Unit,Value_Qualifier,Sample_Fraction,MDL,MDL_Unit,Result_Comment
0,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,"BOD, Biochemical oxygen demand",2.16,mg/l,I,NaN,NaN,NaN,NaN
1,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,"Chlorophyll a, corrected for pheophytin",21.40,ug/l,NaN,NaN,NaN,NaN,NaN
2,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Dissolved oxygen (DO),4.25,mg/l,NaN,NaN,NaN,NaN,NaN
3,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Dissolved oxygen saturation,54.90,percent (%),NaN,NaN,NaN,NaN,NaN
4,SARASOTA_COASTALCREEK_WQ,ALL,SARASOTA_COASTALCREEK_WQ,ALL,7/28/2006 0:00,11:53:00 AM,Sample,Surface,0.5,m,Fecal Coliform,330.00,cfu/100ml,NaN,NaN,NaN,NaN,NaN


In [29]:
nutrients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514087 entries, 0 to 514086
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   DataSource           514087 non-null  object 
 1   StationID            514087 non-null  object 
 2   Actual_DataSource    514087 non-null  object 
 3   Actual_StationID     514087 non-null  object 
 4   Activity_Start_Date  514087 non-null  object 
 5   Activity_Start_Time  432036 non-null  object 
 6   Activity_Type        493753 non-null  object 
 7   RelativeDepth        463443 non-null  object 
 8   Activity_Depth       512176 non-null  float64
 9   Activity_Depth_Unit  511981 non-null  object 
 10  Characteristic       514087 non-null  object 
 11  Result_Value         514087 non-null  float64
 12  Result_Unit          459490 non-null  object 
 13  Value_Qualifier      87876 non-null   object 
 14  Sample_Fraction      190407 non-null  object 
 15  MDL              

In [30]:
nutrients[[
    "Characteristic",
    "Result_Value",
    "Result_Unit"
]].dropna(subset=["Result_Value"]).head(20)

,Characteristic,Result_Value,Result_Unit
0,"BOD, Biochemical oxygen demand",2.160,mg/l
1,"Chlorophyll a, corrected for pheophytin",21.400,ug/l
2,Dissolved oxygen (DO),4.250,mg/l
3,Dissolved oxygen saturation,54.900,percent (%)
4,Fecal Coliform,330.000,cfu/100ml
5,Nitrogen,1.556,mg/l
6,"Nitrogen, ammonia as N",0.245,mg/l
7,"Nitrogen, Kjeldahl",1.530,mg/l
8,"Nitrogen, Nitrite (NO2) + Nitrate (NO3) as N",0.026,mg/l
9,pH,7.910,NaN


Result_Value was not included in the numerical scaling list because it contains measurements for different characteristics and units, making the values unsuitable for being standardized as one numerical feature.

I checked the number of unique values in the categorical columns and found meaningful variation across all eight selected variables, so they were identified as candidates for one-hot encoding.

In [31]:
nutrients_categorical = [
    "Activity_Type",
    "RelativeDepth",
    "Activity_Depth_Unit",
    "Characteristic",
    "Result_Unit",
    "Value_Qualifier",
    "Sample_Fraction",
    "MDL_Unit"
]

nutrients[nutrients_categorical].nunique(dropna=False)

Activity_Type           4
RelativeDepth           4
Activity_Depth_Unit     3
Characteristic         21
Result_Unit             9
Value_Qualifier        27
Sample_Fraction         5
MDL_Unit               16
dtype: int64

In [32]:
nutrients_numeric = [
    "Activity_Depth",
    "MDL"
]

Identified Activity_Depth and MDL as numerical features suitable for scaling. Result_Value was left as a special case because it represents different measurements and units depending on Characteristic and Result_Unit.

Result_Value is a numerical measurement, but scaling is deferred because the column contains different types of measurements with different units.

In [33]:
sarasota.head()

,time,z,air_pressure_at_mean_sea_level,air_pressure_at_mean_sea_level_qc_agg,dew_point_temperature,dew_point_temperature_qc_agg,air_temperature,air_temperature_qc_agg,visibility_in_air,visibility_in_air_qc_agg,wind_speed_of_gust,wind_speed_of_gust_qc_agg,wind_speed,wind_speed_qc_agg,wind_from_direction,wind_from_direction_qc_agg
0,UTC,m,millibars,NaN,degree_Celsius,NaN,degree_Celsius,NaN,m,NaN,m.s-1,NaN,m.s-1,NaN,degrees,NaN
1,2022-07-11T21:53:00Z,0.0,1012.5,2.0,23.9,2.0,30.0,2.0,16093.44,2.0,NaN,2.0,5.1444444444,2.0,230.0,2.0
2,2022-07-11T22:53:00Z,0.0,1012.8,2.0,22.8,2.0,30.0,2.0,16093.44,2.0,9.26,2.0,5.6588888889,2.0,220.0,2.0
3,2022-07-11T23:53:00Z,0.0,1013.1,2.0,23.9,2.0,29.4,2.0,16093.44,2.0,NaN,2.0,5.1444444444,2.0,230.0,2.0
4,2022-07-12T00:53:00Z,0.0,1013.4,2.0,23.3,2.0,28.9,2.0,16093.44,2.0,NaN,2.0,4.63,2.0,240.0,2.0


In [34]:
sarasota.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41286 entries, 0 to 41285
Data columns (total 16 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   time                                   41286 non-null  object 
 1   z                                      41286 non-null  object 
 2   air_pressure_at_mean_sea_level         33675 non-null  object 
 3   air_pressure_at_mean_sea_level_qc_agg  41285 non-null  float64
 4   dew_point_temperature                  41136 non-null  object 
 5   dew_point_temperature_qc_agg           41285 non-null  float64
 6   air_temperature                        41144 non-null  object 
 7   air_temperature_qc_agg                 41285 non-null  float64
 8   visibility_in_air                      41042 non-null  object 
 9   visibility_in_air_qc_agg               41285 non-null  float64
 10  wind_speed_of_gust                     5362 non-null   object 
 11  wi

Removing the units row

In [35]:
sarasota = sarasota.iloc[1:].copy()

In [36]:
sarasota_numeric_cols = [
    "z",
    "air_pressure_at_mean_sea_level",
    "dew_point_temperature",
    "air_temperature",
    "visibility_in_air",
    "wind_speed_of_gust",
    "wind_speed",
    "wind_from_direction"
]

for col in sarasota_numeric_cols:
    sarasota[col] = pd.to_numeric(sarasota[col])

^^ Converting the measurement columns to numbers

This will show how many different values each numerical column contains

In [37]:
sarasota[sarasota_numeric_cols].nunique()

z                                   1
air_pressure_at_mean_sea_level    346
dew_point_temperature              92
air_temperature                   103
visibility_in_air                  24
wind_speed_of_gust                 69
wind_speed                         58
wind_from_direction                40
dtype: int64

Z was constant and excluded, while the weather measurements were selected for scaling. 
Wind direction was treated separately because it is a circular measurement, where 0 degrees and 360 degrees represent the same direction

In [38]:
sarasota["wind_from_direction"].value_counts(dropna=False).head(20)

wind_from_direction
0.0      3336
70.0     1964
80.0     1862
60.0     1648
50.0     1639
90.0     1613
110.0    1604
100.0    1551
40.0     1410
120.0    1356
30.0     1189
270.0    1129
280.0    1125
130.0    1026
NaN       967
320.0     955
260.0     947
240.0     946
330.0     944
250.0     910
Name: count, dtype: int64

In [39]:
sarasota_numeric = [
    "air_pressure_at_mean_sea_level",
    "dew_point_temperature",
    "air_temperature",
    "visibility_in_air",
    "wind_speed_of_gust",
    "wind_speed"
]

In [41]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [ ]:
cyanotoxins_encoded = encoder.fit_transform(
    cyanotoxins[cyanotoxins_categorical]
)

cyanotoxins_encoded_columns = encoder.get_feature_names_out(
    cyanotoxins_categorical
)

cyanotoxins_encoded_df = pd.DataFrame(
    cyanotoxins_encoded,
    columns=cyanotoxins_encoded_columns,
    index=cyanotoxins.index
)

In [ ]:
habsos_encoded = encoder.fit_transform(
    habsos[habsos_categorical]
)

habsos_encoded_columns = encoder.get_feature_names_out(
    habsos_categorical
)

habsos_encoded_df = pd.DataFrame(
    habsos_encoded,
    columns=habsos_encoded_columns,
    index=habsos.index
)

In [ ]:
nutrients_encoded = encoder.fit_transform(
    nutrients[nutrients_categorical]
)

nutrients_encoded_columns = encoder.get_feature_names_out(
    nutrients_categorical
)

nutrients_encoded_df = pd.DataFrame(
    nutrients_encoded,
    columns=nutrients_encoded_columns,
    index=nutrients.index
)

In [40]:
bradenton_scaler = StandardScaler()

bradenton[bradenton_numeric] = bradenton_scaler.fit_transform(
    bradenton[bradenton_numeric]
)

ValueError: could not convert string to float: ' '

In [ ]:
habsos_scaler = StandardScaler()

habsos[habsos_numeric] = habsos_scaler.fit_transform(
    habsos[habsos_numeric]
)

In [ ]:
mote_scaler = StandardScaler()

mote[mote_numeric] = mote_scaler.fit_transform(
    mote[mote_numeric]
)

In [ ]:
nutrients_scaler = StandardScaler()

nutrients[nutrients_numeric] = nutrients_scaler.fit_transform(
    nutrients[nutrients_numeric]
)

In [ ]:
sarasota_scaler = StandardScaler()

sarasota[sarasota_numeric] = sarasota_scaler.fit_transform(
    sarasota[sarasota_numeric]
)

In [ ]:
cyanotoxins_scaler = StandardScaler()

cyanotoxins[cyanotoxins_numeric] = cyanotoxins_scaler.fit_transform(
    cyanotoxins[cyanotoxins_numeric]
)